# US Industry Forecast Based on Macro Leading Indicators
This notebook implements industry trend predictions with macro leading indicators, modeled after `lab/ts_leading_metric_impact_on_sectors.ipynb` but fetching data directly from Fred and yfinance for the specified metrics and ETFs.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
from statsmodels.tsa.api import VAR
import warnings
import os
import urllib.request

warnings.filterwarnings("ignore")


In [ ]:
# configurations
start_date = '2014-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')
n_forecast = 90
n_input_width = 90

fred_metrics = ['M2', 'T10YFF', 'DGS10','SP500', 'UMCSENT',
                'PERMIT', 'NEWY636BPPRIV', 'NYBPPRIVSA',
                'ANDENO', 'ACOGNO', 'AMTMNO', 'ACDGNO', 'DGORDER',
                'AWHMAN', 'AWHAETP', 'AWHAEMAN',
                'DTCDISA066MSFRBNY',
                'ICSA', 'NYICLAIMS', 'NJICLAIMS',
                'MDSP', 'DRSFRMACBS', 'M0264AUSM500NNBR', 'BOGZ1FL153165106Q', 'DCOILWTICO']

etf_list = ['VOX', 'VCR', 'VDC', 'VDE', 'VFH', 'VHT', 'VIS', 'VGT', 'VAW', 'VNQ', 'VPU',
            'QQQ', 'VOO', 'VTV', 'VIGAX', 'VO', 'VB', 'VGK']

macro_dir = 'data/macro/US'
etf_dir = 'data/etf/US'
forecast_dir = 'data/dwa/forecast'

for d in [macro_dir, etf_dir, forecast_dir]:
    os.makedirs(d, exist_ok=True)


In [ ]:
print("Fetching Fred data...")
fred_data = pd.DataFrame()

for metric in fred_metrics:
    try:
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={metric}'
        csv_path = f'{macro_dir}/{metric}.csv'
        urllib.request.urlretrieve(url, csv_path)
        df = pd.read_csv(csv_path, na_values=['.'])
        
        # find the date column
        date_col = [col for col in df.columns if 'date' in col.lower()]
        if date_col:
            date_col = date_col[0]
            df[date_col] = pd.to_datetime(df[date_col])
            df.set_index(date_col, inplace=True)
            
            # The value column is the other column
            val_col = [col for col in df.columns if col != date_col][0]
            df = df[[val_col]]
            df.columns = [metric]
            
            # Filter dates
            df = df.loc[(df.index >= start_date) & (df.index <= end_date)]
            
            if fred_data.empty:
                fred_data = df
            else:
                fred_data = fred_data.merge(df, left_index=True, right_index=True, how='outer')
    except Exception as e:
        print(f"Error fetching {metric}: {e}")
        
fred_data.index.name = 'Date'
print(f"Fetched FRED metrics: {fred_data.columns.tolist()}")


In [ ]:
print("Fetching ETF data...")
etf_data = pd.DataFrame()
for etf in etf_list:
    try:
        df = yf.download(etf, start=start_date, end=end_date, multi_level_index=False, progress=False)
        if not df.empty:
            df.to_csv(f'{etf_dir}/{etf}.csv')
            
            if isinstance(df.columns, pd.MultiIndex):
                close_series = df['Close'][etf].copy()
            else:
                close_series = df['Close'].copy()
                
            close_series.name = etf
            if etf_data.empty:
                etf_data = close_series.to_frame()
            else:
                etf_data = etf_data.merge(close_series, left_index=True, right_index=True, how='outer')
    except Exception as e:
        print(f"Error fetching {etf}: {e}")
        
etf_data.index.name = 'Date'


In [ ]:
print("Concatenating and formatting data...")
# Convert indices to datetime and normalize
etf_data.index = pd.to_datetime(etf_data.index).normalize()
fred_data.index = pd.to_datetime(fred_data.index).normalize()

# Drop any rows where index is NaT
etf_data = etf_data[etf_data.index.notnull()]
fred_data = fred_data[fred_data.index.notnull()]

# Reindex and forward fill FRED data to business daily frequency
full_idx = pd.date_range(start=start_date, end=end_date, freq='B')
fred_data = fred_data.reindex(full_idx).ffill().bfill()
etf_data = etf_data.reindex(full_idx).ffill().bfill()

# Merge dataframes
data_df = etf_data.merge(fred_data, left_index=True, right_index=True, how='inner')

# Some ETFs might fail to download and thus aren't in columns
valid_etfs = [etf for etf in etf_list if etf in data_df.columns]

# Drop NA where all valid ETFs are NaN
data_df = data_df.dropna(subset=valid_etfs, how='all')
data_df = data_df.ffill().bfill()

available_etfs = [c for c in valid_etfs if c in data_df.columns]
available_metrics = [c for c in fred_metrics if c in data_df.columns]
print(f"Data shape after merge: {data_df.shape}")


In [ ]:
print("Building forecast...")
all_forecasts = []

# Build individual VAR model for each ETF
for etf in available_etfs:
    
    # Select features that are numeric
    features = [etf] + available_metrics
    train_df = data_df[features].copy()
    
    # Ensure all data is numeric, convert to float
    train_df = train_df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
    train_df = train_df.ffill().bfill()
    
    # Check for constant columns and drop them (they cause 'not positive definite' errors in VAR)
    train_df = train_df.loc[:, (train_df != train_df.iloc[0]).any()]
    
    # Get actual features
    actual_features = list(train_df.columns)
    if etf not in actual_features:
        print(f"Target {etf} dropped due to being constant or invalid.")
        continue
    
    # Stationarize data by differencing
    diff_df = train_df.diff().dropna()
    
    # Drop highly correlated columns as they can make covariance matrix singular
    corr_matrix = diff_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.98)]
    if etf in to_drop:
        to_drop.remove(etf) # Never drop the target
    diff_df = diff_df.drop(columns=to_drop)
    actual_features = list(diff_df.columns)
    
    # Skip if not enough data or only one variable
    if len(diff_df) < 10 or len(diff_df.columns) < 2:
        print(f"Not enough data for {etf}")
        continue
        
    # Fit VAR model
    model = VAR(diff_df)
    try:
        model_result = model.fit(1)
    except Exception as e:
        print(f"VAR fallback failed for {etf}: {e}")
        continue
            
    # Forecast
    lag_order = model_result.k_ar
    if len(diff_df) < lag_order or lag_order == 0:
        continue
        
    last_obs = diff_df.values[-lag_order:]
    try:
        forecast_diff = model_result.forecast(y=last_obs, steps=n_forecast)
    except Exception as e:
        print(f"Forecasting error for {etf}: {e}")
        continue
    
    etf_idx = actual_features.index(etf)
    etf_diff_pred = forecast_diff[:, etf_idx]
    
    last_val = train_df[etf].iloc[-1]
    etf_pred = last_val + np.cumsum(etf_diff_pred)
    
    last_date = train_df.index[-1]
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=n_forecast, freq='B')
    
    pred_df = pd.DataFrame({
        'Date': future_dates,
        'Ticker': etf,
        'Prediction': etf_pred
    })
    all_forecasts.append(pred_df)

if all_forecasts:
    final_forecast = pd.concat(all_forecasts, ignore_index=True)
    out_path = f'{forecast_dir}/US_industry.csv'
    final_forecast.to_csv(out_path, index=False)
    print(f"Predictions saved to {out_path}")
else:
    print("No forecasts generated.")
